# Data Loading: Reuters-21578 Dataset

Data source: https://github.com/marius92mc/document-classification-reuters21578/tree/master

## Setup

In [1]:
# Check Python version
!python --version

Python 3.13.9


In [3]:
# %pip install torch==2.9.1 torchvision --index-url https://download.pytorch.org/whl/cu128

In [4]:
# Load / save the list of installed libraries
# !pip freeze > new_requirements.txt
# %pip install -r my_requirements.txt

In [4]:
# Import libraries
import numpy as np
import pandas as pd
import re
import os

In [5]:
# Set CUDA_VISIBLE_DEVICES before importing torch
seed = 42

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)


## Handle input data

In [6]:
# Import libraries
import json
from pathlib import Path
from pandas import json_normalize

In [7]:
# Combine .json files into a DataFrame
data_dir = Path('raw/reuters21578/json') 
files = sorted(data_dir.glob('**/*.json'))
print(f'Found {len(files)} json files in {data_dir}')
records = []
for fp in files:
    try:
        with fp.open('r', encoding='utf-8') as fh:
            obj = json.load(fh)
        # Support files that contain a list of objects or a single object
        if isinstance(obj, list):
            records.extend(obj)
        elif isinstance(obj, dict):
            records.append(obj)
        else:
            # skip non-dict/list payloads
            print(f'skipped (unexpected type) {fp}')
    except Exception as e:
        print(f'error reading {fp}: {e}')

print(f'Collected {len(records)} records')
if len(records) == 0:
    df = pd.DataFrame()
else:
    # normalize nested JSON into flat table where possible
    df = json_normalize(records)

print('DataFrame shape:', df.shape)
display(df.head())

Found 22 json files in raw\reuters21578\json
Collected 21578 records
DataFrame shape: (21578, 7)


,title,body,date,topics,places,id,organisations
0,BAHIA COCOA REVIEW,Showers continued throughout the week in\nthe ...,26-FEB-1987 15:01:01.79,[cocoa],"[el-salvador, usa, uruguay]",1,NaN
1,STANDARD OIL <SRD> TO FORM FINANCIAL UNIT,Standard Oil Co and BP North America\nInc said...,26-FEB-1987 15:02:20.00,NaN,[usa],2,NaN
2,TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN,Texas Commerce Bancshares Inc's Texas\nCommerc...,26-FEB-1987 15:03:27.51,NaN,[usa],3,NaN
3,TALKING POINT/BANKAMERICA <BAC> EQUITY OFFER,BankAmerica Corp is not under\npressure to act...,26-FEB-1987 15:07:13.72,NaN,"[usa, brazil]",4,NaN
4,NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESERVE,The U.S. Agriculture Department\nreported the ...,26-FEB-1987 15:10:44.60,"[grain, wheat, corn, barley, oat, sorghum]",[usa],5,NaN


In [8]:
df = df.rename(columns={'body': 'text'})

df

,title,text,date,topics,places,id,organisations
0,BAHIA COCOA REVIEW,Showers continued throughout the week in\nthe ...,26-FEB-1987 15:01:01.79,[cocoa],"[el-salvador, usa, uruguay]",1,NaN
1,STANDARD OIL <SRD> TO FORM FINANCIAL UNIT,Standard Oil Co and BP North America\nInc said...,26-FEB-1987 15:02:20.00,NaN,[usa],2,NaN
2,TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN,Texas Commerce Bancshares Inc's Texas\nCommerc...,26-FEB-1987 15:03:27.51,NaN,[usa],3,NaN
3,TALKING POINT/BANKAMERICA <BAC> EQUITY OFFER,BankAmerica Corp is not under\npressure to act...,26-FEB-1987 15:07:13.72,NaN,"[usa, brazil]",4,NaN
4,NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESERVE,The U.S. Agriculture Department\nreported the ...,26-FEB-1987 15:10:44.60,"[grain, wheat, corn, barley, oat, sorghum]",[usa],5,NaN
...,...,...,...,...,...,...,...
21573,JAPAN/INDIA CONFERENCE CUTS GULF WAR RISK CHARGES,The Japan/India-Pakistan-Gulf/Japan\nshipping ...,19-OCT-1987 00:34:08.94,[ship],"[hong-kong, japan, india, pakistan, iran, iraq]",21574,NaN
21574,SOVIET INDUSTRIAL GROWTH/TRADE SLOWER IN 1987,The Soviet Union's industrial output is\ngrowi...,19-OCT-1987 00:18:22.79,[ipi],[ussr],21575,NaN
21575,SIX KILLED IN SOUTH AFRICAN GOLD MINE ACCIDENT,Six black miners have been killed\nand two inj...,19-OCT-1987 00:05:11.26,[gold],[south-africa],21576,NaN
21576,PROJECTIONS SHOW SWISS VOTERS WANT TRIED PARTIES,The prospect of a dominant alliance of\nsocial...,19-OCT-1987 00:03:21.69,NaN,[switzerland],21577,NaN


In [9]:
df.to_csv('raw/reuters21578.csv', index=False)

In [10]:
# Drop rows where 'text' is missing or empty
df = df.dropna(subset=['text']).reset_index(drop=True)
df = df[df['text'].astype(str).str.strip() != '']
dataset = df
display(dataset.head())
print('DataFrame shape:', dataset.shape)

,title,text,date,topics,places,id,organisations
0,BAHIA COCOA REVIEW,Showers continued throughout the week in\nthe ...,26-FEB-1987 15:01:01.79,[cocoa],"[el-salvador, usa, uruguay]",1,NaN
1,STANDARD OIL <SRD> TO FORM FINANCIAL UNIT,Standard Oil Co and BP North America\nInc said...,26-FEB-1987 15:02:20.00,NaN,[usa],2,NaN
2,TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN,Texas Commerce Bancshares Inc's Texas\nCommerc...,26-FEB-1987 15:03:27.51,NaN,[usa],3,NaN
3,TALKING POINT/BANKAMERICA <BAC> EQUITY OFFER,BankAmerica Corp is not under\npressure to act...,26-FEB-1987 15:07:13.72,NaN,"[usa, brazil]",4,NaN
4,NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESERVE,The U.S. Agriculture Department\nreported the ...,26-FEB-1987 15:10:44.60,"[grain, wheat, corn, barley, oat, sorghum]",[usa],5,NaN


DataFrame shape: (19043, 7)


In [ ]:
# df.to_csv('raw/reuters_drop_na.csv', index=False)

In [12]:
df_drop_na_topics = df.dropna(subset=['topics']).reset_index(drop=True)

df_drop_na_topics

,title,text,date,topics,places,id,organisations
0,BAHIA COCOA REVIEW,Showers continued throughout the week in\nthe ...,26-FEB-1987 15:01:01.79,[cocoa],"[el-salvador, usa, uruguay]",1,NaN
1,NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESERVE,The U.S. Agriculture Department\nreported the ...,26-FEB-1987 15:10:44.60,"[grain, wheat, corn, barley, oat, sorghum]",[usa],5,NaN
2,ARGENTINE 1986/87 GRAIN/OILSEED REGISTRATIONS,Argentine grain board figures show\ncrop regis...,26-FEB-1987 15:14:36.41,"[veg-oil, linseed, lin-oil, soy-oil, sun-oil, ...",[argentina],6,NaN
3,CHAMPION PRODUCTS <CH> APPROVES STOCK SPLIT,Champion Products Inc said its\nboard of direc...,26-FEB-1987 15:17:11.20,[earn],[usa],9,NaN
4,COMPUTER TERMINAL SYSTEMS <CPML> COMPLETES SALE,Computer Terminal Systems Inc said\nit has com...,26-FEB-1987 15:18:06.67,[acq],[usa],10,NaN
...,...,...,...,...,...,...,...
10372,N.Z.'S CHASE CORP MAKES OFFER FOR ENTREGROWTH,Chase Corp Ltd <CHCA.WE> said it will\nmake an...,19-OCT-1987 01:35:27.64,[acq],[new-zealand],21571,NaN
10373,TOKYO DEALERS SEE DOLLAR POISED TO BREACH 140 YEN,Tokyo's foreign exchange market is watching\nn...,19-OCT-1987 00:59:58.56,"[money-fx, dlr, yen]","[japan, west-germany, usa]",21573,NaN
10374,JAPAN/INDIA CONFERENCE CUTS GULF WAR RISK CHARGES,The Japan/India-Pakistan-Gulf/Japan\nshipping ...,19-OCT-1987 00:34:08.94,[ship],"[hong-kong, japan, india, pakistan, iran, iraq]",21574,NaN
10375,SOVIET INDUSTRIAL GROWTH/TRADE SLOWER IN 1987,The Soviet Union's industrial output is\ngrowi...,19-OCT-1987 00:18:22.79,[ipi],[ussr],21575,NaN


In [13]:
df_drop_na_topics = df_drop_na_topics.drop_duplicates(subset=['text']).reset_index(drop=True)

df_drop_na_topics

,title,text,date,topics,places,id,organisations
0,BAHIA COCOA REVIEW,Showers continued throughout the week in\nthe ...,26-FEB-1987 15:01:01.79,[cocoa],"[el-salvador, usa, uruguay]",1,NaN
1,NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESERVE,The U.S. Agriculture Department\nreported the ...,26-FEB-1987 15:10:44.60,"[grain, wheat, corn, barley, oat, sorghum]",[usa],5,NaN
2,ARGENTINE 1986/87 GRAIN/OILSEED REGISTRATIONS,Argentine grain board figures show\ncrop regis...,26-FEB-1987 15:14:36.41,"[veg-oil, linseed, lin-oil, soy-oil, sun-oil, ...",[argentina],6,NaN
3,CHAMPION PRODUCTS <CH> APPROVES STOCK SPLIT,Champion Products Inc said its\nboard of direc...,26-FEB-1987 15:17:11.20,[earn],[usa],9,NaN
4,COMPUTER TERMINAL SYSTEMS <CPML> COMPLETES SALE,Computer Terminal Systems Inc said\nit has com...,26-FEB-1987 15:18:06.67,[acq],[usa],10,NaN
...,...,...,...,...,...,...,...
10247,N.Z.'S CHASE CORP MAKES OFFER FOR ENTREGROWTH,Chase Corp Ltd <CHCA.WE> said it will\nmake an...,19-OCT-1987 01:35:27.64,[acq],[new-zealand],21571,NaN
10248,TOKYO DEALERS SEE DOLLAR POISED TO BREACH 140 YEN,Tokyo's foreign exchange market is watching\nn...,19-OCT-1987 00:59:58.56,"[money-fx, dlr, yen]","[japan, west-germany, usa]",21573,NaN
10249,JAPAN/INDIA CONFERENCE CUTS GULF WAR RISK CHARGES,The Japan/India-Pakistan-Gulf/Japan\nshipping ...,19-OCT-1987 00:34:08.94,[ship],"[hong-kong, japan, india, pakistan, iran, iraq]",21574,NaN
10250,SOVIET INDUSTRIAL GROWTH/TRADE SLOWER IN 1987,The Soviet Union's industrial output is\ngrowi...,19-OCT-1987 00:18:22.79,[ipi],[ussr],21575,NaN


In [14]:
df_drop_na_topics.to_csv('complete/reuters.csv', index=False)

In [17]:
# Processing with data_cleaning.py


In [18]:
df_text = pd.read_parquet(f'processed/parquet/reuters_text.parquet')

In [19]:
df_text

,text
0,Showers continued throughout the week in the B...
1,The U.S. Agriculture Department reported the f...
2,Argentine grain board figures show crop regist...
3,Champion Products Inc said its board of direct...
4,Computer Terminal Systems Inc said it has comp...
...,...
10247,Chase Corp Ltd <CHCA.WE> said it will make an ...
10248,Tokyo's foreign exchange market is watching ne...
10249,The Japan/India-Pakistan-Gulf/Japan shipping c...
10250,The Soviet Union's industrial output is growin...


In [20]:
# Print dataset statistics
avg_chars = df_text['text'].apply(lambda x: len(str(x))).mean()
print("Average number of characters for text passage:", avg_chars)
avg_words = df_text['text'].apply(lambda x: len(x.split(" "))).mean()
print("Average number of words for text passage:", avg_words)
print("Average word length:", avg_chars / avg_words)

Average number of characters for text passage: 797.0564767850176
Average number of words for text passage: 133.72327350760827
Average word length: 5.960491811768791


In [21]:
# Convert 'body' column into list containing documents
docs = df_text['text'].tolist()

In [22]:
# Ensure all documents are strings
for i in range(len(docs)):
    docs[i] = str(docs[i])

In [23]:
# Print length of the document list
print("Number of documents:", len(docs))

Number of documents: 10252


In [24]:
# Print a sample of document
print(docs[9])

Dean Foods Co expects earnings for the fourth quarter ending May 30 to exceed those of the same year-ago period, Chairman Kenneth Douglas told analysts. In the fiscal 1986 fourth quarter the food processor reported earnings of 40 cts a share. Douglas also said the year's sales should exceed 1.4 billion dlrs, up from 1.27 billion dlrs the prior year. He repeated an earlier projection that third-quarter earnings "will probably be off slightly" from last year's 40 cts a share, falling in the range of 34 cts to 36 cts a share. Douglas said it was too early to project whether the anticipated fourth quarter performance would be "enough for us to exceed the prior year's overall earnings" of 1.53 dlrs a share. In 1988, Douglas said Dean should experience "a 20 pct improvement in our bottom line from effects of the tax reform act alone." President Howard Dean said in fiscal 1988 the company will derive benefits of various dairy and frozen vegetable acquisitions from Ryan Milk to the Larsen Co. 

In [9]:
import ast

In [10]:
# Create a safe conversion function
def safe_convert(val):
    # If it's already a list, just return it
    if isinstance(val, list):
        return val
    # If it's empty/NaN, return an empty list (which tomotopy needs for unlabeled data)
    if pd.isna(val) or val == "":
        return []
    # Otherwise, safely evaluate the string
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        # Fallback just in case the string is malformed
        return []

In [11]:
df['topics'] = df['topics'].apply(safe_convert)

In [13]:
labels_df = df[['topics']]

labels_df

,topics
0,[cocoa]
1,"[grain, wheat, corn, barley, oat, sorghum]"
2,"[veg-oil, linseed, lin-oil, soy-oil, sun-oil, ..."
3,[earn]
4,[acq]
...,...
10372,[acq]
10373,"[money-fx, dlr, yen]"
10374,[ship]
10375,[ipi]


In [14]:
labels_df = labels_df.rename(columns={'topics': 'labels'})

labels_df

,labels
0,[cocoa]
1,"[grain, wheat, corn, barley, oat, sorghum]"
2,"[veg-oil, linseed, lin-oil, soy-oil, sun-oil, ..."
3,[earn]
4,[acq]
...,...
10372,[acq]
10373,"[money-fx, dlr, yen]"
10374,[ship]
10375,[ipi]


In [29]:
labels_df.to_parquet('processed/reuters_drop_na_topics_labels.parquet', index=False)

In [75]:
df_clean['text'] = df_clean['text'].astype(str)

C:\Users\zheng\AppData\Local\Temp\ipykernel_23396\2758870227.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['text'] = df_clean['text'].astype(str)


In [38]:
df_remove_na_topics = df_to_save.dropna(subset=['topics'])

df_remove_na_topics

,text,topics
0,Showers continued throughout the week in\nthe ...,[cocoa]
4,The U.S. Agriculture Department\nreported the ...,"[grain, wheat, corn, barley, oat, sorghum]"
5,Argentine grain board figures show\ncrop regis...,"[veg-oil, linseed, lin-oil, soy-oil, sun-oil, ..."
8,Champion Products Inc said its\nboard of direc...,[earn]
9,Computer Terminal Systems Inc said\nit has com...,[acq]
...,...,...
21570,Chase Corp Ltd <CHCA.WE> said it will\nmake an...,[acq]
21572,Tokyo's foreign exchange market is watching\nn...,"[money-fx, dlr, yen]"
21573,The Japan/India-Pakistan-Gulf/Japan\nshipping ...,[ship]
21574,The Soviet Union's industrial output is\ngrowi...,[ipi]


In [39]:
df_remove_na_topics.to_csv('reuters_no_na_topics.csv', index=False)

In [16]:
# Count documents with missing topics
df['topics'].isnull().sum()

np.int64(8666)

In [17]:
# Count documents with missing organisations
df['organisations'].isnull().sum()

np.int64(18189)

In [18]:
# Print dataset statistics
avg_chars = dataset['body'].apply(lambda x: len(str(x))).mean()
print("Average number of characters for text passage:", avg_chars)
avg_words = dataset['body'].apply(lambda x: len(x.split(" "))).mean()
print("Average number of words for text passage:", avg_words)
print("Average word length:", avg_chars / avg_words)

Average number of characters for text passage: 836.4228850496245
Average number of words for text passage: 139.83101402090006
Average word length: 5.981669309246426


In [19]:
# Convert 'body' column into list containing documents
docs = dataset['body'].tolist()

In [20]:
# Ensure all documents are strings
for i in range(len(docs)):
    docs[i] = str(docs[i])

In [21]:
# Print length of the document list
print("Number of documents:", len(docs))

Number of documents: 19043


In [22]:
# Print a sample of document
print(docs[9])

Computer Terminal Systems Inc said
it has completed the sale of 200,000 shares of its common
stock, and warrants to acquire an additional one mln shares, to
<Sedio N.V.> of Lugano, Switzerland for 50,000 dlrs.
    The company said the warrants are exercisable for five
years at a purchase price of .125 dlrs per share.
    Computer Terminal said Sedio also has the right to buy
additional shares and increase its total holdings up to 40 pct
of the Computer Terminal's outstanding common stock under
certain circumstances involving change of control at the
company.
    The company said if the conditions occur the warrants would
be exercisable at a price equal to 75 pct of its common stock's
market price at the time, not to exceed 1.50 dlrs per share.
    Computer Terminal also said it sold the technolgy rights to
its Dot Matrix impact technology, including any future
improvements, to <Woodco Inc> of Houston, Tex. for 200,000
dlrs. But, it said it would continue to be the exclusive
worldwide l

In [23]:
# Load dataframe from .csv file
df_new = pd.read_csv('reuters21578.csv')

df_new.head()

,title,body,date,topics,places,id,organisations
0,BAHIA COCOA REVIEW,Showers continued throughout the week in\nthe ...,26-FEB-1987 15:01:01.79,['cocoa'],"['el-salvador', 'usa', 'uruguay']",1,NaN
1,STANDARD OIL <SRD> TO FORM FINANCIAL UNIT,Standard Oil Co and BP North America\nInc said...,26-FEB-1987 15:02:20.00,NaN,['usa'],2,NaN
2,TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN,Texas Commerce Bancshares Inc's Texas\nCommerc...,26-FEB-1987 15:03:27.51,NaN,['usa'],3,NaN
3,TALKING POINT/BANKAMERICA <BAC> EQUITY OFFER,BankAmerica Corp is not under\npressure to act...,26-FEB-1987 15:07:13.72,NaN,"['usa', 'brazil']",4,NaN
4,NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESERVE,The U.S. Agriculture Department\nreported the ...,26-FEB-1987 15:10:44.60,"['grain', 'wheat', 'corn', 'barley', 'oat', 's...",['usa'],5,NaN


In [24]:
df_new.shape

(19043, 7)

In [31]:
reuters_drop_na = pd.read_csv('processed/reuters_drop_na.csv')

reuters_drop_na

,text
0,Showers continued throughout the week in the B...
1,Standard Oil Co and BP North America Inc said ...
2,Texas Commerce Bancshares Inc's Texas Commerce...
3,BankAmerica Corp is not under pressure to act ...
4,The U.S. Agriculture Department reported the f...
...,...
19038,The Japan/India-Pakistan-Gulf/Japan shipping c...
19039,The Soviet Union's industrial output is growin...
19040,Six black miners have been killed and two inju...
19041,The prospect of a dominant alliance of sociali...


In [38]:
reuters_drop_na['text'][5667]

"Consolidated Rail Corp said it and CSX Corp's CSX Transportation Corp subsidiary have reduced rates on boxcar shipments of manufactured products moving between CSX points in the South and Conrail points in the Northeast and Midwest. Conrail said the prices, which were effective March one, are expected to meet or beat comparable truck transportation prices in virtually all target markets. It said the reduced prices also represent a simplification of the prvious tariff-based rate structure, which used price and milage scales to determine rates. The new prices are constructed from a single table of origin and destination Zip Codes."

In [33]:
reuters_drop_na_topics = pd.read_csv('processed/reuters_drop_na_topics.csv')

reuters_drop_na_topics

,text
0,Showers continued throughout the week in the B...
1,The U.S. Agriculture Department reported the f...
2,Argentine grain board figures show crop regist...
3,Champion Products Inc said its board of direct...
4,Computer Terminal Systems Inc said it has comp...
...,...
10372,Chase Corp Ltd <CHCA.WE> said it will make an ...
10373,Tokyo's foreign exchange market is watching ne...
10374,The Japan/India-Pakistan-Gulf/Japan shipping c...
10375,The Soviet Union's industrial output is growin...


In [12]:
df_clean.shape

(10377, 1)

In [16]:
labels = df['topics'].tolist()

labels

[['cocoa'],
 ['grain', 'wheat', 'corn', 'barley', 'oat', 'sorghum'],
 ['veg-oil',
  'linseed',
  'lin-oil',
  'soy-oil',
  'sun-oil',
  'soybean',
  'oilseed',
  'corn',
  'sunseed',
  'grain',
  'sorghum',
  'wheat'],
 ['earn'],
 ['acq'],
 ['earn'],
 ['earn', 'acq'],
 ['earn'],
 ['earn'],
 ['earn'],
 ['wheat', 'grain'],
 ['copper'],
 ['earn'],
 ['earn'],
 ['earn'],
 ['housing'],
 ['earn'],
 ['earn'],
 ['earn'],
 ['earn'],
 ['earn'],
 ['coffee'],
 ['acq', 'ship'],
 ['acq'],
 ['sugar'],
 ['trade'],
 ['reserves'],
 ['ship'],
 ['earn'],
 ['earn'],
 ['earn'],
 ['grain', 'corn'],
 ['money-supply'],
 ['ship'],
 ['earn'],
 ['earn'],
 ['earn'],
 ['acq'],
 ['veg-oil', 'soybean', 'oilseed', 'meal-feed', 'soy-meal'],
 ['earn'],
 ['earn'],
 ['coffee'],
 ['money-supply'],
 ['money-supply'],
 ['earn'],
 ['earn'],
 ['earn'],
 ['earn'],
 ['earn'],
 ['earn'],
 ['earn'],
 ['acq'],
 ['grain', 'wheat', 'corn', 'oat', 'rye', 'sorghum', 'soybean', 'oilseed'],
 ['earn'],
 ['money-supply'],
 ['cotton'],
 ['su

In [16]:
import ast

clean_data = []

for item in labels:
    # Safely convert the string "['acq, yen']" into a real Python list ['acq, yen']
    evaluated_list = ast.literal_eval(item)
    
    for item in evaluated_list:
        clean_data.append(item)

print(clean_data)
# Output: ['cocoa', 'acq', 'yen', 'sugar', 'coffee']

['cocoa', 'grain', 'wheat', 'corn', 'barley', 'oat', 'sorghum', 'veg-oil', 'linseed', 'lin-oil', 'soy-oil', 'sun-oil', 'soybean', 'oilseed', 'corn', 'sunseed', 'grain', 'sorghum', 'wheat', 'earn', 'acq', 'earn', 'earn', 'acq', 'earn', 'earn', 'earn', 'wheat', 'grain', 'copper', 'earn', 'earn', 'earn', 'housing', 'earn', 'earn', 'earn', 'earn', 'earn', 'coffee', 'acq', 'ship', 'acq', 'sugar', 'trade', 'reserves', 'ship', 'earn', 'earn', 'earn', 'grain', 'corn', 'money-supply', 'ship', 'earn', 'earn', 'earn', 'acq', 'veg-oil', 'soybean', 'oilseed', 'meal-feed', 'soy-meal', 'earn', 'earn', 'coffee', 'money-supply', 'money-supply', 'earn', 'earn', 'earn', 'earn', 'earn', 'earn', 'earn', 'acq', 'grain', 'wheat', 'corn', 'oat', 'rye', 'sorghum', 'soybean', 'oilseed', 'earn', 'money-supply', 'cotton', 'sugar', 'grain', 'ship', 'earn', 'money-supply', 'acq', 'money-supply', 'earn', 'carcass', 'livestock', 'earn', 'grain', 'acq', 'earn', 'crude', 'acq', 'earn', 'acq', 'acq', 'grain', 'earn', 'e

In [26]:
nan_count = 0

for i in range(len(labels)):
    if labels[i] is np.nan:
        nan_count += 1

print("Number of NaN labels:", nan_count)

Number of NaN labels: 8666


In [27]:
import ast

In [17]:
dict_labels = dict()

for i in range(len(labels)):

    if(labels[i] is not np.nan):
        label = labels[i]

        for j in range(len(label)):
            if label[j] not in dict_labels:
                dict_labels[label[j]] = 1
            else:
                dict_labels[label[j]] += 1


print("Unique labels:", list(dict_labels.keys()))
print("Label counts:", list(dict_labels.values()))


Unique labels: ['cocoa', 'grain', 'wheat', 'corn', 'barley', 'oat', 'sorghum', 'veg-oil', 'linseed', 'lin-oil', 'soy-oil', 'sun-oil', 'soybean', 'oilseed', 'sunseed', 'earn', 'acq', 'copper', 'housing', 'coffee', 'ship', 'sugar', 'trade', 'reserves', 'money-supply', 'meal-feed', 'soy-meal', 'rye', 'cotton', 'carcass', 'livestock', 'crude', 'nat-gas', 'cpi', 'gnp', 'money-fx', 'interest', 'bop', 'rice', 'red-bean', 'rubber', 'copra-cake', 'palm-oil', 'palmkernel', 'tea', 'plywood', 'alum', 'gold', 'platinum', 'strategic-metal', 'tapioca', 'tin', 'rapeseed', 'groundnut-oil', 'rape-oil', 'cornglutenfeed', 'citruspulp', 'rape-meal', 'wool', 'dlr', 'l-cattle', 'retail', 'ipi', 'silver', 'iron-steel', 'hog', 'propane', 'heat', 'gas', 'jobs', 'lei', 'yen', 'saudriyal', 'zinc', 'orange', 'pet-chem', 'fuel', 'wpi', 'potato', 'lead', 'groundnut', 'can', 'fishmeal', 'income', 'palladium', 'nickel', 'lumber', 'jet', 'instal-debt', 'dfl', 'dmk', 'stg', 'coconut-oil', 'corn-oil', 'inventories', 'cpu

In [23]:
np.average(list(dict_labels.values()))

np.float64(110.26890756302521)

In [24]:
print(dict_labels)

{'cocoa': 68, 'grain': 574, 'wheat': 287, 'corn': 224, 'barley': 48, 'oat': 13, 'sorghum': 34, 'veg-oil': 136, 'linseed': 2, 'lin-oil': 2, 'soy-oil': 25, 'sun-oil': 8, 'soybean': 111, 'oilseed': 182, 'sunseed': 17, 'earn': 3776, 'acq': 2210, 'copper': 77, 'housing': 18, 'coffee': 143, 'ship': 295, 'sugar': 175, 'trade': 515, 'reserves': 73, 'money-supply': 126, 'meal-feed': 50, 'soy-meal': 26, 'rye': 2, 'cotton': 62, 'carcass': 75, 'livestock': 112, 'crude': 566, 'nat-gas': 126, 'cpi': 101, 'gnp': 153, 'money-fx': 684, 'interest': 424, 'bop': 101, 'rice': 67, 'red-bean': 1, 'rubber': 49, 'copra-cake': 3, 'palm-oil': 42, 'palmkernel': 3, 'tea': 15, 'plywood': 4, 'alum': 58, 'gold': 133, 'platinum': 11, 'strategic-metal': 32, 'tapioca': 4, 'tin': 33, 'rapeseed': 35, 'groundnut-oil': 2, 'rape-oil': 8, 'cornglutenfeed': 2, 'citruspulp': 1, 'rape-meal': 1, 'wool': 2, 'dlr': 168, 'l-cattle': 9, 'retail': 23, 'ipi': 57, 'silver': 36, 'iron-steel': 65, 'hog': 26, 'propane': 6, 'heat': 25, 'gas

In [22]:
df_clean.shape[0]

10377

In [25]:
label_sum = 0

for key, value in sorted(dict_labels.items(), key=lambda x: x[1]):
    label_sum += value

print("Total label count:", label_sum)
print("Avg label count", label_sum / df_clean.shape[0])

Total label count: 13122
Avg label count 1.264527320034692


In [26]:
for key, value in sorted(dict_labels.items(), key=lambda x: x[1]): 
    print("{} : {}".format(key, value))

red-bean : 1
citruspulp : 1
rape-meal : 1
corn-oil : 1
peseta : 1
ringgit : 1
castorseed : 1
rupiah : 1
skr : 1
dkr : 1
lin-meal : 1
cottonseed : 1
bfr : 1
hk : 1
linseed : 2
lin-oil : 2
rye : 2
groundnut-oil : 2
cornglutenfeed : 2
wool : 2
fishmeal : 2
cpu : 2
castor-oil : 2
nkr : 2
sun-meal : 2
copra-cake : 3
palmkernel : 3
saudriyal : 3
can : 3
palladium : 3
dfl : 3
cotton-oil : 3
rand : 3
pork-belly : 3
lit : 3
f-cattle : 3
sfr : 3
plywood : 4
tapioca : 4
austdlr : 4
nzdlr : 4
instal-debt : 5
propane : 6
potato : 6
inventories : 6
coconut : 6
coconut-oil : 7
naphtha : 7
sun-oil : 8
rape-oil : 8
jet : 8
l-cattle : 9
groundnut : 10
platinum : 11
nickel : 11
income : 12
oat : 13
lei : 13
dmk : 14
tea : 15
sunseed : 17
lumber : 17
stg : 17
housing : 18
retail : 23
soy-oil : 25
heat : 25
orange : 25
soy-meal : 26
hog : 26
fuel : 28
wpi : 29
strategic-metal : 32
tin : 33
sorghum : 34
rapeseed : 35
lead : 35
silver : 36
pet-chem : 41
palm-oil : 42
zinc : 43
barley : 48
rubber : 49
meal-fe

In [28]:
len(dict_labels.items())

119